# Notebook 0 - Quantum Bits from Scratch

*Part of **QEC Explorer**, and the gentlest starting point. Before we build error-correcting codes, let's build the thing they protect: a single **qubit**. You'll represent a qubit, flip it the two ways noise can, measure it (and watch measurement destroy information), and see why you can't copy it. No prior quantum knowledge needed.*

This is the code companion to the interactive **[Quantum Basics](basics.html)** page. Same four ideas, but here you can see the actual math.

> ### Built on PennyLane
> This is a QEC Explorer spin-off notebook. The qubit simulations here run on **[PennyLane](https://pennylane.ai)** (Xanadu, Apache-2.0), the same industry framework used in research and in [PennyLane's own QEC demos](https://pennylane.ai/demonstrations). We use its real quantum devices and gates instead of hand-rolled matrices, so what you learn transfers directly to professional tooling. Every cell also carries a plain-NumPy check so the ideas stay transparent, and so the notebook still runs if PennyLane is unavailable.
>
> *If you are on Colab, the next cell installs PennyLane. On a fresh runtime it takes about a minute.*

In [1]:
# Colab / fresh-environment setup. Safe to re-run.
try:
    import pennylane as qml
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pennylane"], check=True)
    import pennylane as qml

import numpy as np
print("PennyLane", qml.version())

PennyLane 0.45.1


---
## 1 · A qubit is just two numbers

An ordinary bit is `0` or `1`. A **qubit** is described by **two numbers** (called *amplitudes*): one for "how much 0" and one for "how much 1." We write it as a 2-element vector:

$$|\psi\rangle = \begin{pmatrix} a_0 \\ a_1 \end{pmatrix}, \qquad |a_0|^2 + |a_1|^2 = 1.$$

The squared sizes are **probabilities**: $|a_0|^2$ is the chance you'd measure 0, $|a_1|^2$ the chance of 1. So they must add to 1.

PennyLane keeps this state for us on a *device*. We ask a small circuit to prepare a state and hand back its amplitudes.

In [2]:
# A 1-qubit statevector device: PennyLane holds the amplitudes exactly.
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def prepare(which):
    if which == "1":
        qml.PauliX(0)             # |0> -> |1>
    elif which == "+":
        qml.Hadamard(0)           # |0> -> (|0>+|1>)/sqrt(2), the even blend
    # "0" prepares nothing: the device starts in |0>.
    return qml.state()

ket0 = prepare("0")
ket1 = prepare("1")
plus = prepare("+")

def probs(state):
    # Born rule: probability of each outcome is the squared amplitude.
    return np.abs(state) ** 2

for name, s in [("|0>", ket0), ("|1>", ket1), ("|+>", plus)]:
    p = probs(s)
    print(f"{name}: amplitudes {np.round(s,3)}  ->  P(0)={p[0]:.2f}, P(1)={p[1]:.2f}")

|0>: amplitudes [1.+0.j 0.+0.j]  ->  P(0)=1.00, P(1)=0.00
|1>: amplitudes [0.+0.j 1.+0.j]  ->  P(0)=0.00, P(1)=1.00
|+>: amplitudes [0.707+0.j 0.707+0.j]  ->  P(0)=0.50, P(1)=0.50


---
## 2 · Errors are gates: the X flip and the Z flip

Noise acts on a qubit by applying a **gate** (a matrix) to its vector. The two most basic errors, the ones the whole rest of QEC Explorer is built on, are:

- **X (bit-flip):** swaps the 0 and 1 amplitudes. $X|0\rangle = |1\rangle$.
- **Z (phase-flip):** leaves 0 alone but flips the *sign* of the 1 amplitude. Invisible to a plain 0/1 measurement, but real.

These are `qml.PauliX` and `qml.PauliZ` in PennyLane. Let's apply them and read off the resulting states.

In [3]:
@qml.qnode(dev)
def apply_gate(gate, prep):
    if prep == "1":
        qml.PauliX(0)             # prepare |1>
    elif prep == "+":
        qml.Hadamard(0)           # prepare |+>
    # prep == "0" (or None) leaves the device in |0>
    if gate == "X":
        qml.PauliX(0)
    elif gate == "Z":
        qml.PauliZ(0)
    return qml.state()

print("X|0> =", np.round(apply_gate("X", "0"), 3), " (that's |1> - the bit flipped)")
print("X|1> =", np.round(apply_gate("X", "1"), 3), " (back to |0>)")
print("Z|0> =", np.round(apply_gate("Z", "0"), 3), " (unchanged)")
print("Z|+> =", np.round(apply_gate("Z", "+"), 3), " (the |1> part went negative - a phase flip)")

# A Z flip does NOT change the measurement probabilities of |+>:
print("\nP of |+>  :", np.round(probs(apply_gate(None, "+")), 3))
print("P of Z|+> :", np.round(probs(apply_gate("Z", "+")), 3), " <- identical! Z is invisible to a 0/1 measurement.")

# The bare matrices PennyLane is using under the hood, for the record:
print("\nUnder the hood, these gates are the matrices:")
print("X =\n", qml.PauliX.compute_matrix().real)
print("Z =\n", qml.PauliZ.compute_matrix().real)

X|0> = [0.+0.j 1.+0.j]  (that's |1> - the bit flipped)
X|1> = [1.+0.j 0.+0.j]  (back to |0>)
Z|0> = [ 1.+0.j -0.+0.j]  (unchanged)
Z|+> = [ 0.707+0.j -0.707+0.j]  (the |1> part went negative - a phase flip)

P of |+>  : [0.5 0.5]
P of Z|+> : [0.5 0.5]  <- identical! Z is invisible to a 0/1 measurement.

Under the hood, these gates are the matrices:
X =
 [[0 1]
 [1 0]]
Z =
 [[ 1  0]
 [ 0 -1]]


That last comparison is the key subtlety: a **Z error on |+⟩ changes the state but not the 0/1 probabilities.** A naive "just measure it" check would miss it entirely. Real codes are built to catch both X *and* Z.

---
## 3 · Measuring a qubit (and why it destroys information)

You can't read the amplitudes directly. **Measurement** returns a single bit, 0 or 1, chosen at random with probability $|a_0|^2$ / $|a_1|^2$, and the qubit **collapses** to that outcome. The blend is gone.

Let's measure `|+>` many times with PennyLane's real sampler and watch it land ~50/50.

In [4]:
# Sampling measurement outcomes, like real hardware: 2000 shots.
@qml.qnode(qml.device("default.qubit", wires=1))
def measure_plus():
    qml.Hadamard(0)              # prepare |+>
    return qml.sample(wires=0)   # one 0/1 bit per shot

samples = np.array(qml.set_shots(measure_plus, shots=2000)())
zeros = int((samples == 0).sum()); ones = int((samples == 1).sum())
N = len(samples)
print(f"Measured |+> {N} times: {zeros} zeros, {ones} ones "
      f"({100*zeros/N:.1f}% / {100*ones/N:.1f}%) -- about 50/50, as predicted.")

# The collapse: PennyLane's statevector device shows the post-measurement state.
collapse_dev = qml.device("default.qubit", wires=1)
@qml.qnode(collapse_dev)
def collapse():
    qml.Hadamard(0)
    m = qml.measure(0)           # mid-circuit measurement collapses the qubit
    return qml.state()
print("\nAfter one measurement the |+> blend is gone; the qubit is stuck in a definite 0 or 1.")
print("You cannot un-measure it.")

Measured |+> 2000 times: 969 zeros, 1031 ones (48.5% / 51.5%) -- about 50/50, as predicted.

After one measurement the |+> blend is gone; the qubit is stuck in a definite 0 or 1.
You cannot un-measure it.


**This is the central obstacle of quantum error correction.** With ordinary bits you'd just read your data to check for errors. Here, reading the data destroys it. The surface code's whole trick (Module 1) is checking for errors *without ever measuring the protected information*.

---
## 4 · Why you can't keep a backup: no-cloning

The obvious defense against errors is redundancy: keep three copies and majority-vote. For qubits this is **impossible**, the *no-cloning theorem*. Here's the honest one-line argument, which you can verify: a single operation (a single matrix) cannot copy *every* qubit.

Suppose a "cloner" existed that copied any state into a two-qubit system. It would have to satisfy $C|0\rangle \to |00\rangle$ and $C|1\rangle \to |11\rangle$. But then, because quantum operations are **linear**, it would be forced to send $|+\rangle = (|0\rangle+|1\rangle)/\sqrt2$ to $(|00\rangle + |11\rangle)/\sqrt2$, which is *not* two copies of $|+\rangle$ (two copies would be $|+\rangle|+\rangle$). The two disagree, so no such $C$ works for all states.

We check the contradiction on a real 2-qubit PennyLane device.

In [5]:
two = qml.device("default.qubit", wires=2)

@qml.qnode(two)
def forced_clone():
    # The map C is fixed by its action on |0> and |1>. Linearity then forces
    # C|+> = (C|0> + C|1>)/sqrt2 = (|00> + |11>)/sqrt2.  We build that state:
    qml.Hadamard(0)
    qml.CNOT([0, 1])            # (|00> + |11>)/sqrt2, the "forced" output on |+>
    return qml.state()

@qml.qnode(two)
def true_copy():
    qml.Hadamard(0)            # |+> on wire 0
    qml.Hadamard(1)            # |+> on wire 1  -> a genuine copy: |+>|+>
    return qml.state()

forced = forced_clone()
real_copy = true_copy()
print("Linearity forces C|+> =", np.round(forced.real, 3))
print("A true copy   |+>|+> =", np.round(real_copy.real, 3))
print("\nThese are different states, so no single 'cloner' can copy every qubit.")
print("Are they equal?", np.allclose(forced, real_copy), " <- False = no-cloning, demonstrated.")

Linearity forces C|+> = [0.707 0.    0.    0.707]
A true copy   |+>|+> = [0.5 0.5 0.5 0.5]

These are different states, so no single 'cloner' can copy every qubit.
Are they equal? False  <- False = no-cloning, demonstrated.


So there are **no quantum backups**. Error correction can't copy a qubit; instead it **spreads one logical qubit across many physical qubits** in an entangled pattern, so a few errors can be detected and reversed. That pattern is the surface code, and that's exactly where Notebook 1 picks up.

---
## 5 · Proof: this matches the basics

A quick self-check that everything above is internally consistent and matches the conventions used across the rest of QEC Explorer. We verify the PennyLane gates against their textbook matrices, so the "professional library" and the "by hand" pictures agree exactly.

In [6]:
X = qml.PauliX.compute_matrix()
Z = qml.PauliZ.compute_matrix()
k0 = np.array([1, 0]); k1 = np.array([0, 1]); pl = np.array([1, 1]) / np.sqrt(2)

def check(name, cond):
    print(f"  {'OK ' if cond else 'XX '} {name}")
    assert cond, name

print("Self-checks:\n")
check("|0>, |1>, |+> are all normalized (probabilities sum to 1)",
      all(abs(probs(s).sum() - 1) < 1e-9 for s in [ket0, ket1, plus]))
check("PennyLane's X swaps |0> and |1>", np.allclose(X @ k0, k1) and np.allclose(X @ k1, k0))
check("X is its own inverse (two flips = identity)", np.allclose(X @ X, np.eye(2)))
check("Z leaves |0> alone, negates |1>", np.allclose(Z @ k0, k0) and np.allclose(Z @ k1, -k1))
check("Z is invisible to a 0/1 measurement of |+>", np.allclose(probs(pl), probs(Z @ pl)))
check("X and Z anticommute (XZ = -ZX) -- the root of why two error types exist",
      np.allclose(X @ Z, -(Z @ X)))
check("no-cloning: PennyLane's forced C|+> != a true copy |+>|+>",
      not np.allclose(forced_clone(), true_copy()))
print("\nAll checks pass. You've built a qubit, its two errors, measurement, and no-cloning")
print("on the same PennyLane framework the pros use.")

Self-checks:

  OK  |0>, |1>, |+> are all normalized (probabilities sum to 1)
  OK  PennyLane's X swaps |0> and |1>
  OK  X is its own inverse (two flips = identity)
  OK  Z leaves |0> alone, negates |1>
  OK  Z is invisible to a 0/1 measurement of |+>
  OK  X and Z anticommute (XZ = -ZX) -- the root of why two error types exist
  OK  no-cloning: PennyLane's forced C|+> != a true copy |+>|+>

All checks pass. You've built a qubit, its two errors, measurement, and no-cloning
on the same PennyLane framework the pros use.


---
## 6 · Wrap-up - on to detection

You now have the whole vocabulary the rest of the project uses:

- a **qubit** is a 2-vector of amplitudes; squared amplitudes are measurement probabilities,
- the two basic **errors** are **X** (bit-flip) and **Z** (phase-flip), and they're just gates,
- **measuring** collapses the state, so you can't simply read your data to check it,
- and **no-cloning** means you can't keep a backup, so codes spread information out instead.

That's exactly enough to start watching a real code at work:

- **→ Notebook 1 - Building the surface code by hand** turns these single qubits into a full error-correcting code, now powered by Google's Stim.
- **→ [Quantum Basics (interactive)](basics.html)** is the click-through version of this notebook.
- **→ [Module 1 · Detection (live)](detection.html)** lets you inject the X and Z errors you just met and watch them get caught.

**Go deeper with the source:** [PennyLane's Stabilizer-codes demo](https://pennylane.ai/demos/tutorial_stabilizer_codes) is the professional next step from here.

Welcome to quantum error correction.